# ParaForge CUDA on Google Colab

Builds and runs the naive and the shared-memory tiled CUDA matmul (`src/matmul_cuda.cu`) on a
Colab GPU, checks both against the CPU and compares them on the same inputs.

1. **Runtime > Change runtime type > T4 GPU** (any NVIDIA GPU works), then run the cells in order.
2. Colab ships `nvcc`, CMake and a C++ compiler; nothing else to install.
3. Numbers come from whatever GPU Colab assigns; record the GPU name with them.

In [ ]:
!nvidia-smi --query-gpu=name,driver_version,compute_cap --format=csv
!nvcc --version | tail -n 2
!cmake --version | head -n 1

Get the code. Set `REPO_URL` to your clone URL, **or** leave it empty and upload a zip of the
repository (Files panel, or the prompt below) named `MiniGPU_Runtime.zip`.

In [ ]:
REPO_URL = ""  # e.g. "https://github.com/<user>/MiniGPU_Runtime.git"
import os, zipfile
if REPO_URL:
    !git clone --depth 1 {REPO_URL} /content/MiniGPU_Runtime
else:
    if not os.path.exists("/content/MiniGPU_Runtime.zip"):
        from google.colab import files
        files.upload()  # choose MiniGPU_Runtime.zip
    zipfile.ZipFile("/content/MiniGPU_Runtime.zip").extractall("/content")
# os.walk is top-down, so this is the outermost folder with a CMakeLists.txt (zips may nest one level).
root = next(d for d, _, f in os.walk("/content") if "CMakeLists.txt" in f)
%cd {root}

In [ ]:
!cmake -S . -B build -DCMAKE_BUILD_TYPE=Release -DPARAFORGE_ENABLE_CUDA=ON
!cmake --build build -j 4

In [ ]:
# CPU tests plus paraforge_cuda_tests (naive and tiled kernels vs CPU reference: tiny,
# odd, partial-tile, square and rectangular shapes).
!ctest --test-dir build --output-on-failure

Benchmark: `paraforge_cuda_bench [m] [k] [n] [repeats]` multiplies an `m x k` by a `k x n`
matrix with each kernel (same inputs), checks every result against the CPU reference and prints
one CSV row per kernel (`cuda_naive`, `cuda_tiled`) with medians of `repeats` calls.
`kernel_ms` is kernel-only (CUDA events); `h2d_ms` / `d2h_ms` are the copies; `end_to_end_ms`
is wall time of the whole call including device alloc/free.

In [ ]:
!./build/paraforge_cuda_bench 1024 1024 1024 5
!./build/paraforge_cuda_bench 2048 512 1024 5 | tail -n +2
!./build/paraforge_cuda_bench 1000 700 300 5 | tail -n +2   # no dimension a multiple of 16
!./build/paraforge_cuda_bench 2048 2048 2048 5 | tail -n +2